# 🌲 Diagnóstico — Cadastro Nacional de Florestas Públicas 2024

**Objetivo:** Inspecionar completamente o CNFP 2024 e os arquivos auxiliares  
antes de definir a arquitetura do WebMap e do repositório GitHub.  

**Fluxo:** Diagnóstico → Processamento → WebMap → GitHub Pages


In [ ]:
# =============================================================================
#  CÉLULA 1 — DEPENDÊNCIAS E CAMINHOS
# =============================================================================

import os, sys, warnings, unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

BASE_GEO = Path(r'.../data')
BASE_OUT = Path(r'.../outputs')
BASE_OUT.mkdir(parents=True, exist_ok=True)

SHP_CNFP = BASE_GEO / 'cnfp_2024' / 'cnfp_2024.shp'
CST_CNFP = BASE_GEO / 'cnfp_2024' / 'cnfp_2024.cst'
PRJ_CNFP = BASE_GEO / 'cnfp_2024' / 'cnfp_2024.prj'
SHP_MUN  = BASE_GEO / 'BR_Municipios_2024' / 'BR_Municipios_2024.shp'
SHP_UF   = BASE_GEO / 'BR_UF_2024' / 'BR_UF_2024.shp'

CRS_PROJ = 'EPSG:5641'
CRS_GEO  = 'EPSG:4674'

SEP  = '─' * 72
SEP2 = '═' * 72

def ok(m):   print(f'  ✔  {m}')
def warn(m): print(f'  ⚠  {m}')
def info(m): print(f'  →  {m}')
def sec(t):  print(f'\n{SEP2}\n  {t}\n{SEP2}')
def sub(t):  print(f'\n  ┌─ {t}')

print(f'  Python   : {sys.version.split()[0]}')
print(f'  GeoPandas: {gpd.__version__}')
print(f'  Pandas   : {pd.__version__}')
print(f'  Data     : {datetime.now().strftime("%d/%m/%Y %H:%M")}')
ok('Dependências carregadas.')


In [ ]:
# =============================================================================
#  BLOCO 1 · INVENTÁRIO DE ARQUIVOS
# =============================================================================
sec('BLOCO 1 · INVENTÁRIO DE ARQUIVOS')

files_check = [
    (SHP_CNFP,                                   'CNFP 2024 (.shp)'),
    (BASE_GEO / 'cnfp_2024' / 'cnfp_2024.dbf',  'CNFP 2024 (.dbf)'),
    (BASE_GEO / 'cnfp_2024' / 'cnfp_2024.shx',  'CNFP 2024 (.shx)'),
    (CST_CNFP,                                   'CNFP 2024 (.cst) — legenda'),
    (PRJ_CNFP,                                   'CNFP 2024 (.prj) — projeção'),
    (SHP_MUN,                                    'Municípios Brasil 2024'),
    (SHP_UF,                                     'Estados Brasil 2024'),
]

total_mb = 0
for path, label in files_check:
    if path.exists():
        mb = path.stat().st_size / (1024**2)
        total_mb += mb
        print(f'  ✔  {label:<45} {mb:8.2f} MB')
    else:
        print(f'  ✗  {label:<45}  NÃO ENCONTRADO')

print(f'\n  Total: {total_mb:.1f} MB')


In [ ]:
# =============================================================================
#  BLOCO 2 · ARQUIVO .CST — LEGENDA DE CATEGORIAS
# =============================================================================
sec('BLOCO 2 · ARQUIVO .CST — LEGENDA DE CATEGORIAS')

cst_content = ''
if CST_CNFP.exists():
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:
            with open(CST_CNFP, encoding=enc, errors='replace') as f:
                cst_content = f.read()
            ok(f'Lido com encoding: {enc}')
            break
        except Exception as e:
            warn(f'Falhou com {enc}: {e}')
    print('\n  Conteúdo do .cst:')
    print('  ' + SEP)
    print(cst_content)
else:
    warn('.cst não encontrado — categorias serão inferidas dos dados.')

# PRJ
sec('BLOCO 3 · ARQUIVO .PRJ — SISTEMA DE REFERÊNCIA')
if PRJ_CNFP.exists():
    with open(PRJ_CNFP, encoding='utf-8', errors='replace') as f:
        prj_content = f.read()
    print(f'  {prj_content}')
else:
    warn('.prj não encontrado.')


In [ ]:
# =============================================================================
#  BLOCO 4 · SHAPEFILE CNFP 2024 — INSPEÇÃO COMPLETA
# =============================================================================
sec('BLOCO 4 · SHAPEFILE CNFP 2024 — INSPEÇÃO COMPLETA')

print('  Carregando CNFP 2024...')
gdf = gpd.read_file(SHP_CNFP)
ok(f'Carregado: {len(gdf):,} feições')

sub('Informações básicas')
print(f'     Feições         : {len(gdf):,}')
print(f'     CRS             : {gdf.crs}')
print(f'     EPSG            : {gdf.crs.to_epsg() if gdf.crs else "?"}')
print(f'     Geometria       : {gdf.geometry.geom_type.unique().tolist()}')
print(f'     Bbox            : {gdf.total_bounds.round(4).tolist()}')
print(f'     Colunas ({len(gdf.columns)}): {list(gdf.columns)}')

sub('Tipos de dado por coluna')
for col in gdf.columns:
    if col == 'geometry': continue
    dtype  = gdf[col].dtype
    nuniq  = gdf[col].nunique()
    nulls  = gdf[col].isna().sum()
    sample = str(gdf[col].dropna().unique()[:4].tolist())[:80]
    print(f'     {col:<30} {str(dtype):<12} unique={nuniq:<6} '
          f'nulls={nulls:<5} ex: {sample}')


In [ ]:
# =============================================================================
#  BLOCO 5 · CATEGORIAS  |  BLOCO 6 · ÁREAS
# =============================================================================
sec('BLOCO 5 · ANÁLISE DE CATEGORIAS')

cat_cols = [c for c in gdf.columns
            if c != 'geometry' and gdf[c].dtype == object
            and 1 < gdf[c].nunique() < 80]

for col in cat_cols:
    sub(f"Distribuição — '{col}'")
    vc = gdf[col].value_counts(dropna=False)
    max_n = vc.max()
    for val, cnt in vc.items():
        bar = '█' * min(int(cnt / max_n * 30), 30)
        pct = cnt / len(gdf) * 100
        print(f'     {str(val):<50} {bar:<30}  {cnt:>6,} ({pct:5.1f}%)')

sub('Estatísticas das colunas numéricas')
num_cols = [c for c in gdf.columns if c != 'geometry'
            and gdf[c].dtype in ['int32','int64','float32','float64']]
for col in num_cols:
    s = gdf[col].dropna()
    if len(s) == 0: continue
    print(f'     {col:<30} min={s.min():.3f}  max={s.max():.3f}  '
          f'média={s.mean():.3f}  soma={s.sum():.2f}')

# ── Áreas ─────────────────────────────────────────────────────────────────────
sec('BLOCO 6 · ANÁLISE DE ÁREA')

area_cols = [c for c in gdf.columns
             if any(k in c.lower() for k in ['area','ha','km','hectare'])]
if area_cols:
    info(f'Colunas de área existentes: {area_cols}')
    for ac in area_cols:
        try:
            v = gdf[ac].dropna().astype(float)
            print(f'     {ac}: soma={v.sum():,.2f}  média={v.mean():,.4f}  '
                  f'max={v.max():,.2f}  min={v.min():,.6f}')
        except Exception:
            pass

info('Calculando área geométrica via EPSG:5641 (Albers IBGE)...')
gdf_proj = gdf.to_crs(CRS_PROJ)
gdf['area_calc_ha']  = (gdf_proj.geometry.area / 10_000).round(4)
gdf['area_calc_km2'] = (gdf_proj.geometry.area / 1_000_000).round(6)

total_ha  = gdf['area_calc_ha'].sum()
total_km2 = gdf['area_calc_km2'].sum()
br_area   = 851_576_700   # ha
pct_br    = total_ha / br_area * 100

print(f'\n  Área total calculada : {total_ha:>18,.2f} ha')
print(f'  Área total calculada : {total_km2:>18,.2f} km²')
print(f'  % do território BR   : {pct_br:>17.2f}%')

sub('Histograma de tamanhos (ha)')
bins   = [0, 100, 1_000, 10_000, 100_000, 1_000_000, float('inf')]
labels = ['<100 ha','100–1k','1k–10k','10k–100k','100k–1M','>1M ha']
gdf['size_class'] = pd.cut(gdf['area_calc_ha'], bins=bins, labels=labels)
for cls, cnt in gdf['size_class'].value_counts().sort_index().items():
    pct = cnt / len(gdf) * 100
    bar = '█' * min(int(pct), 30)
    print(f'     {str(cls):<15} {bar:<30}  {cnt:>6,} ({pct:5.1f}%)')

sub('Top 10 maiores feições')
id_cols = [c for c in gdf.columns
           if any(k in c.lower() for k in ['nome','name','nm_','descr','id','cod'])
           and c != 'geometry'][:4]
print(gdf.nlargest(10,'area_calc_ha')[id_cols+['area_calc_ha','area_calc_km2']]
      .to_string(index=False))


In [ ]:
# =============================================================================
#  BLOCO 7 · DISTRIBUIÇÃO ESTADUAL
# =============================================================================
sec('BLOCO 7 · DISTRIBUIÇÃO ESTADUAL E REGIONAL')

uf_cols = [c for c in gdf.columns
           if any(k in c.lower() for k in ['uf','estado','sigla'])]
info(f'Colunas candidatas de UF: {uf_cols}')

if uf_cols:
    uf_col  = uf_cols[0]
    uf_area = (gdf.groupby(uf_col)
               .agg(n=('area_calc_ha','count'), area_ha=('area_calc_ha','sum'))
               .sort_values('area_ha', ascending=False)
               .reset_index())
    total = uf_area['area_ha'].sum()
    print(f'\n  {uf_col:<6} {"Feições":>8} {"Área (ha)":>16} {"% BR":>8}  Barra')
    print(f'  {SEP}')
    for _, r in uf_area.iterrows():
        pct = r['area_ha'] / total * 100
        bar = '█' * min(int(pct), 35)
        print(f'  {str(r[uf_col]):<6} {int(r["n"]):>8,} '
              f'{r["area_ha"]:>16,.2f}  {pct:>7.2f}%  {bar}')

    if cat_cols:
        sub(f'Área por UF × Categoria ({cat_cols[0]})')
        pivot = (gdf.groupby([uf_col, cat_cols[0]])['area_calc_ha']
                 .sum().unstack(fill_value=0))
        print(pivot.round(0).to_string())
else:
    info('Nenhuma coluna de UF — fazendo spatial join com shapefile de estados...')
    gdf_uf = gpd.read_file(SHP_UF).to_crs(CRS_GEO)[['SIGLA_UF','NM_UF','geometry']]
    gdf_geo = gdf.to_crs(CRS_GEO)
    joined  = gpd.sjoin(gdf_geo, gdf_uf, how='left', predicate='intersects')
    uf_area = (joined.groupby('SIGLA_UF')
               .agg(n=('area_calc_ha','count'), area_ha=('area_calc_ha','sum'))
               .sort_values('area_ha', ascending=False).reset_index())
    total = uf_area['area_ha'].sum()
    for _, r in uf_area.iterrows():
        pct = r['area_ha'] / total * 100
        bar = '█' * min(int(pct), 35)
        print(f'  {str(r["SIGLA_UF"]):<6} {int(r["n"]):>8,} '
              f'{r["area_ha"]:>16,.2f}  {pct:>7.2f}%  {bar}')


In [ ]:
# =============================================================================
#  BLOCO 8 · QUALIDADE GEOMÉTRICA  |  BLOCO 9 · CRS
# =============================================================================
sec('BLOCO 8 · QUALIDADE GEOMÉTRICA')

n_invalid = (~gdf.geometry.is_valid).sum()
n_empty   = gdf.geometry.is_empty.sum()
n_null    = gdf.geometry.isna().sum()

print(f'     Geometrias inválidas : {n_invalid:,}')
print(f'     Geometrias vazias    : {n_empty:,}')
print(f'     Geometrias nulas     : {n_null:,}')

if n_invalid > 0:
    warn(f'{n_invalid} inválidas — make_valid() será aplicado no processamento.')
else:
    ok('Todas as geometrias são válidas.')

sub('Tipos de geometria')
for gt, cnt in gdf.geometry.geom_type.value_counts().items():
    print(f'     {gt:<25}: {cnt:,}')

# ── CRS ───────────────────────────────────────────────────────────────────────
sec('BLOCO 9 · COMPATIBILIDADE DE CRS')

layers_crs = [
    ('CNFP 2024',         gdf.crs),
    ('Municípios 2024',   gpd.read_file(SHP_MUN, rows=1).crs),
    ('Estados 2024',      gpd.read_file(SHP_UF,  rows=1).crs),
]
print(f'  {"Camada":<25} {"EPSG":>7}  {"Projetado":>10}  CRS resumo')
print(f'  {SEP}')
epsgs = set()
for label, crs in layers_crs:
    epsg   = crs.to_epsg() if crs else '?'
    is_p   = crs.is_projected if crs else False
    resumo = str(crs)[:55] if crs else 'Indefinido'
    compat = '✔' if epsg in [4326, 4674, 4618] else '⚠'
    epsgs.add(epsg)
    print(f'  {compat} {label:<25} {str(epsg):>7}  {"Sim" if is_p else "Não":>10}  {resumo}')

if len(epsgs) == 1:
    ok(f'CRS uniforme: EPSG:{list(epsgs)[0]}')
else:
    info(f'EPSGs distintos: {epsgs} — reprojeção necessária.')


In [ ]:
# =============================================================================
#  BLOCO 10 · INSPEÇÃO VISUAL — MAPA RÁPIDO
# =============================================================================
sec('BLOCO 10 · INSPEÇÃO VISUAL — MAPA RÁPIDO')

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0d1a10')

# Fundo
gdf_uf_plot = gpd.read_file(SHP_UF).to_crs(CRS_GEO)

# Painel esquerdo — mapa
ax1 = axes[0]
ax1.set_facecolor('#0d1a10')
gdf_uf_plot.plot(ax=ax1, facecolor='#112015', edgecolor='#2a4a2e', linewidth=0.5)

cat_col = cat_cols[0] if cat_cols else None
if cat_col:
    cats      = [c for c in gdf[cat_col].dropna().unique()]
    palette   = ['#4ade80','#f59e0b','#60a5fa','#f87171',
                 '#a78bfa','#34d399','#fb923c','#e879f9']
    color_map = {cat: palette[i % len(palette)] for i, cat in enumerate(cats)}
    colors    = gdf[cat_col].map(color_map).fillna('#555')
    gdf.to_crs(CRS_GEO).plot(ax=ax1, color=colors, alpha=0.75,
                               edgecolor='none')
    patches = [mpatches.Patch(color=c, label=str(k))
               for k, c in color_map.items()]
    ax1.legend(handles=patches, loc='lower left', fontsize=6,
               facecolor='#0d1a10', labelcolor='white',
               edgecolor='#2a4a2e', title=cat_col, title_fontsize=7)
else:
    gdf.to_crs(CRS_GEO).plot(ax=ax1, color='#4ade80', alpha=0.7, edgecolor='none')
    color_map = {}

ax1.set_title('CNFP 2024 — Florestas Públicas do Brasil',
              color='#bbf7d0', fontsize=12, fontweight='bold', pad=10)
ax1.axis('off')

# Painel direito — barras por categoria
ax2 = axes[1]
ax2.set_facecolor('#0d1a10')

if cat_col:
    area_by_cat = (gdf.groupby(cat_col)['area_calc_ha']
                   .sum().sort_values(ascending=True))
    bar_colors  = [color_map.get(k, '#4ade80') for k in area_by_cat.index]
    bars = ax2.barh(range(len(area_by_cat)),
                    area_by_cat.values / 1_000_000,
                    color=bar_colors, alpha=0.85, edgecolor='none')
    ax2.set_yticks(range(len(area_by_cat)))
    ax2.set_yticklabels(area_by_cat.index, color='#86efac', fontsize=8)
    ax2.set_xlabel('Área (milhões de ha)', color='#86efac', fontsize=9)
    ax2.set_title('Área por Categoria (M ha)',
                  color='#bbf7d0', fontsize=11, fontweight='bold', pad=10)
    ax2.tick_params(colors='#3d6b4a')
    for sp in ax2.spines.values(): sp.set_color('#1a3320')
    for bar, val in zip(bars, area_by_cat.values):
        ax2.text(bar.get_width() + 0.02,
                 bar.get_y() + bar.get_height()/2,
                 f'{val/1e6:.2f}M ha',
                 va='center', color='#86efac', fontsize=7)
else:
    ax2.text(0.5, 0.5, 'Coluna categórica\nnão identificada',
             ha='center', va='center', color='#86efac',
             fontsize=12, transform=ax2.transAxes)
    ax2.axis('off')

plt.tight_layout(pad=2)
map_path = BASE_OUT / 'diagnostico_cnfp_mapa.png'
plt.savefig(map_path, dpi=150, bbox_inches='tight', facecolor='#0d1a10')
plt.show()
ok(f'Mapa salvo: {map_path}')


In [ ]:
# =============================================================================
#  BLOCO 11 · AMOSTRA  |  BLOCO 12 · SUMÁRIO E OPORTUNIDADES
# =============================================================================
sec('BLOCO 11 · AMOSTRA DOS DADOS')

cols_show = [c for c in gdf.columns if c not in ['geometry','size_class']]
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 35)
print(gdf[cols_show].head(10).to_string(index=False))

sub('Valores ausentes por coluna')
nulls = gdf[cols_show].isna().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    for col, n in nulls.items():
        print(f'     {col:<30}: {n:,} nulos ({n/len(gdf)*100:.1f}%)')
else:
    ok('Nenhum valor ausente.')

# ── Sumário ───────────────────────────────────────────────────────────────────
sec('BLOCO 12 · SUMÁRIO E OPORTUNIDADES ANALÍTICAS')

print(f'''
  ┌──────────────────────────────────────────────────────────────────────┐
  │  RESUMO — CNFP 2024                                                  │
  ├──────────────────────────────────────────────────────────────────────┤
  │  Total de feições       : {len(gdf):>10,}                            │
  │  Área total calculada   : {total_ha:>16,.2f} ha                      │
  │  Área total calculada   : {total_km2:>16,.2f} km²                    │
  │  % do território BR     : {pct_br:>15.2f}%                           │
  │  CRS original           : {str(gdf.crs.to_epsg() if gdf.crs else "?")}                             │
  │  Geometrias inválidas   : {n_invalid:>10,}                           │
  └──────────────────────────────────────────────────────────────────────┘

  OPORTUNIDADES ANALÍTICAS:

  [A] CHOROPLETH ESTADUAL — área e % do território por UF
  [B] RANKING DE MUNICÍPIOS — spatial join CNFP × BR_Municipios
  [C] ANÁLISE POR CATEGORIA — distribuição espacial por tipo
  [D] INDICADORES NACIONAIS — % do Brasil coberto, fragmentação
  [E] WEBMAP GITHUB PAGES — HTML estático + Leaflet.js

  ⚡ Cole o output → definimos juntos o scripts.ipynb!
''')

gdf[cols_show].describe(include='all').to_csv(
    BASE_OUT / 'diagnostico_cnfp_describe.csv')
ok(f'Describe salvo: {BASE_OUT / "diagnostico_cnfp_describe.csv"}')
print(f'  DIAGNÓSTICO CONCLUÍDO — {datetime.now().strftime("%d/%m/%Y %H:%M")}')
